# Training
Nachat Jatusripitak

In [2]:
import xgboost as xgb
import pandas as pd
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
import shap

In [48]:
df = pd.read_csv('dataset_1_v3.csv')
df["date"] = pd.to_datetime(df["date"])
df['month'] = df['date'].dt.month
df['week'] = pd.to_datetime(df['date']).dt.isocalendar().week
df = df.sort_values(['date'])

In [24]:
def rolling_window_cv(train_days, gap_days, test_days, df, model):
    dates = df["date"].unique()
    train_start = 0
    num_days = len(dates)
    i = 1
    errors = []

    while True:
        train_end = train_start + train_days
        test_start = train_end + gap_days
        test_end = test_start + test_days

        if test_end > num_days:
            break

        train_dates = dates[train_start: train_end]
        test_dates = dates[test_start: test_end]

        print(f'Fold {i}')
        print(train_dates.min(), train_dates.max())
        print(test_dates.min(), test_dates.max())

        train_data =  df[df["date"].isin(train_dates)]
        test_data = df[df["date"].isin(test_dates)]

        X_train = train_data.drop(['date', 'delta_pm25_t+1'], axis=1)
        y_train = train_data['delta_pm25_t+1']

        X_test = test_data.drop(['date', 'delta_pm25_t+1'], axis=1)
        y_test = test_data['delta_pm25_t+1']
        
        model.fit(X_train, y_train)

        y_preds = model.predict(X_test)

        mse = mean_squared_error(y_test, y_preds)
        rmse = root_mean_squared_error(y_test, y_preds)
        mae = mean_absolute_error(y_test, y_preds)
        r2 = r2_score(y_test, y_preds)

        errors.append(rmse)

        print(f"MSE: {mse}")
        print(f"RMSE: {rmse}")
        print(f"MAE: {mae}")
        print(f"R^2: {r2}")
        train_start += test_days
        i += 1

    print(np.mean(errors))

In [58]:
model = xgb.XGBRegressor(eta=0.1)

train = df[df["date"].dt.year < 2022]
test = df[df["date"].dt.year == 2022]

rolling_window_cv(train_days=800, gap_days=30, test_days=60, df=train.drop('week', axis=1), model=model)

Fold 1
2018-07-07 00:00:00 2020-09-29 00:00:00
2020-10-30 00:00:00 2020-12-28 00:00:00
MSE: 19.06786609718002
RMSE: 4.36667677956361
MAE: 2.3915940324109117
R^2: 0.4324378719488523
Fold 2
2018-09-05 00:00:00 2020-11-28 00:00:00
2020-12-29 00:00:00 2021-02-26 00:00:00
MSE: 69.04008866800042
RMSE: 8.30903656677478
MAE: 6.437332866319375
R^2: 0.1766057709125134
Fold 3
2018-11-04 00:00:00 2021-01-27 00:00:00
2021-02-27 00:00:00 2021-05-09 00:00:00
MSE: 144.07379443823578
RMSE: 12.003074374435734
MAE: 8.006765877275864
R^2: 0.17418012579053
Fold 4
2019-01-03 00:00:00 2021-04-09 00:00:00
2021-05-10 00:00:00 2021-07-08 00:00:00
MSE: 2.9541216565209862
RMSE: 1.7187558455234375
MAE: 1.257563131214763
R^2: 0.12900799817707387
Fold 5
2019-03-06 00:00:00 2021-06-08 00:00:00
2021-07-09 00:00:00 2021-09-06 00:00:00
MSE: 0.8859732966849917
RMSE: 0.9412615453129867
MAE: 0.7224158187881913
R^2: 0.07579496477967429
Fold 6
2019-05-05 00:00:00 2021-08-07 00:00:00
2021-09-07 00:00:00 2021-11-05 00:00:00
MS

In [59]:
model = DummyRegressor(strategy='constant', constant=0)

train = df[df["date"].dt.year < 2022]
test = df[df["date"].dt.year == 2022]

rolling_window_cv(train_days=800, gap_days=30, test_days=60, df=train.drop('week', axis=1), model=model)

Fold 1
2018-07-07 00:00:00 2020-09-29 00:00:00
2020-10-30 00:00:00 2020-12-28 00:00:00
MSE: 33.61639324563519
RMSE: 5.797964577818252
MAE: 2.586536762734426
R^2: -0.0006044509992977343
Fold 2
2018-09-05 00:00:00 2020-11-28 00:00:00
2020-12-29 00:00:00 2021-02-26 00:00:00
MSE: 84.3918896601449
RMSE: 9.186505846084511
MAE: 7.052120627940899
R^2: -0.00648472892472296
Fold 3
2018-11-04 00:00:00 2021-01-27 00:00:00
2021-02-27 00:00:00 2021-05-09 00:00:00
MSE: 174.5709271100537
RMSE: 13.212529171587615
MAE: 8.399387156491457
R^2: -0.0006270857845565025
Fold 4
2019-01-03 00:00:00 2021-04-09 00:00:00
2021-05-10 00:00:00 2021-07-08 00:00:00
MSE: 3.396715482982265
RMSE: 1.843018036531999
MAE: 1.326595000818509
R^2: -0.0014861817267994848
Fold 5
2019-03-06 00:00:00 2021-06-08 00:00:00
2021-07-09 00:00:00 2021-09-06 00:00:00
MSE: 0.9587098171057102
RMSE: 0.9791372820527825
MAE: 0.7436377670140044
R^2: -8.029993627278742e-05
Fold 6
2019-05-05 00:00:00 2021-08-07 00:00:00
2021-09-07 00:00:00 2021-11